In [1]:
import sys
import pandas as pd
import numpy as np

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]
Pandas: 2.3.3
NumPy: 2.3.5


In [2]:
from pathlib import Path

# Reproducibility: однаковий seed = однакові дані при повторному запуску
rng = np.random.default_rng(42)

N_USERS = 150_000

# Папка CloudPlay/data
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

# Діапазон реєстрацій
start_date = np.datetime64("2024-01-01")
end_date = np.datetime64("2026-08-31")

date_range = (end_date - start_date).astype(int)

# Можливі значення
countries = [
    "US", "DE", "GB", "FR", "PL",
    "UA", "CA", "BR", "JP", "KR",
    "ES", "IT", "NL", "SE", "GE"
]

country_prob = [
    0.18, 0.10, 0.09, 0.07, 0.07,
    0.07, 0.07, 0.06, 0.06, 0.05,
    0.04, 0.04, 0.03, 0.03, 0.04
]

channels = [
    "organic",
    "google_ads",
    "youtube",
    "affiliate",
    "referral",
    "tiktok",
    "display"
]

channel_prob = [
    0.28,
    0.22,
    0.14,
    0.10,
    0.10,
    0.09,
    0.07
]

devices = [
    "windows",
    "macos",
    "android",
    "ios",
    "smart_tv",
    "browser"
]

device_prob = [
    0.48,
    0.10,
    0.17,
    0.10,
    0.07,
    0.08
]

# Створюємо таблицю
users = pd.DataFrame({
    "user_id": np.arange(1, N_USERS + 1),

    "signup_date":
        start_date +
        rng.integers(0, date_range + 1, N_USERS).astype("timedelta64[D]"),

    "country":
        rng.choice(countries, N_USERS, p=country_prob),

    "acquisition_channel":
        rng.choice(channels, N_USERS, p=channel_prob),

    "signup_device":
        rng.choice(devices, N_USERS, p=device_prob),

    "birth_year":
        np.clip(
            rng.normal(1995, 9, N_USERS).round(),
            1955,
            2008
        ).astype(int)
})

# Зберігаємо CSV
users.to_csv(
    DATA_DIR / "users.csv",
    index=False
)

print("Rows:", len(users))
print("File created:", DATA_DIR / "users.csv")

users.head(10)

Rows: 150000
File created: ..\data\users.csv


,user_id,signup_date,country,acquisition_channel,signup_device,birth_year
0,1,2024-03-27,GE,organic,windows,2007
1,2,2026-01-23,GB,organic,windows,2000
2,3,2025-09-29,NL,google_ads,windows,1996
3,4,2025-03-03,IT,google_ads,windows,2008
4,5,2025-02-25,ES,referral,android,1984
5,6,2026-04-16,US,google_ads,windows,2003
6,7,2024-03-24,BR,organic,browser,2000
7,8,2025-11-10,FR,youtube,android,1988
8,9,2024-07-15,PL,affiliate,smart_tv,1991
9,10,2024-04-01,US,organic,windows,2008


In [3]:
print("Shape:", users.shape)

print("\nMissing values:")
print(users.isna().sum())

print("\nDuplicate user_id:")
print(users["user_id"].duplicated().sum())

print("\nSignup range:")
print(users["signup_date"].min(), "→", users["signup_date"].max())

print("\nAcquisition channels:")
print(users["acquisition_channel"].value_counts(normalize=True).round(3))

print("\nCountries:")
print(users["country"].value_counts().head())

Shape: (150000, 6)

Missing values:
user_id                0
signup_date            0
country                0
acquisition_channel    0
signup_device          0
birth_year             0
dtype: int64

Duplicate user_id:
0

Signup range:
2024-01-01 00:00:00 → 2026-08-31 00:00:00

Acquisition channels:
acquisition_channel
organic       0.281
google_ads    0.220
youtube       0.140
referral      0.100
affiliate     0.100
tiktok        0.090
display       0.069
Name: proportion, dtype: float64

Countries:
country
US    26995
DE    15104
GB    13496
CA    10605
PL    10541
Name: count, dtype: int64


In [4]:
# -----------------------------
# SUBSCRIPTIONS
# -----------------------------

N_SUBSCRIPTIONS = 250_000

# Приховані характеристики користувачів.
# Вони потрібні ТІЛЬКИ генератору і не потраплять у users.csv.
user_profile = pd.DataFrame({
    "user_id": users["user_id"],
    "engagement_propensity": rng.normal(0, 1, N_USERS),
    "churn_propensity": rng.normal(0, 1, N_USERS)
})

# Частина характеристик залежить від acquisition channel.
# Точні ефекти залишаємо частиною генератора, а не аналітичного dataset.
profile = users[
    ["user_id", "signup_date", "acquisition_channel"]
].merge(
    user_profile,
    on="user_id"
)

profile.loc[
    profile["acquisition_channel"].isin(["organic", "referral"]),
    "engagement_propensity"
] += 0.25

profile.loc[
    profile["acquisition_channel"].isin(["display", "tiktok"]),
    "churn_propensity"
] += 0.18


# ------------------------------------------------
# Вибираємо користувачів, які матимуть subscriptions
# ------------------------------------------------

# Спочатку гарантуємо велику кількість унікальних subscribers.
first_subscribers = rng.choice(
    users["user_id"],
    size=120_000,
    replace=False
)

# Решта subscription periods можуть належати тим самим users.
repeat_subscribers = rng.choice(
    first_subscribers,
    size=N_SUBSCRIPTIONS - len(first_subscribers),
    replace=True
)

subscription_users = np.concatenate([
    first_subscribers,
    repeat_subscribers
])

rng.shuffle(subscription_users)


subscriptions = pd.DataFrame({
    "subscription_id": np.arange(1, N_SUBSCRIPTIONS + 1),
    "user_id": subscription_users
})


# Додаємо signup_date та hidden characteristics
subscriptions = subscriptions.merge(
    profile,
    on="user_id",
    how="left"
)


# ------------------------------------------------
# Subscription start
# ------------------------------------------------

# Більшість користувачів конвертується відносно скоро,
# але частина — значно пізніше.
days_to_subscription = np.clip(
    rng.exponential(scale=55, size=N_SUBSCRIPTIONS).astype(int),
    0,
    600
)

subscriptions["subscription_start_date"] = (
    subscriptions["signup_date"] +
    pd.to_timedelta(days_to_subscription, unit="D")
)

DATA_END = pd.Timestamp("2026-08-31")

# Не дозволяємо subscriptions виходити за межі dataset
subscriptions["subscription_start_date"] = (
    subscriptions["subscription_start_date"]
    .clip(upper=DATA_END)
)


# ------------------------------------------------
# Plan
# ------------------------------------------------

subscriptions["plan"] = rng.choice(
    ["priority", "ultimate"],
    size=N_SUBSCRIPTIONS,
    p=[0.72, 0.28]
)


# ------------------------------------------------
# Churn probability
# ------------------------------------------------

logit = (
    -0.10
    + 0.50 * subscriptions["churn_propensity"]
    - 0.40 * subscriptions["engagement_propensity"]
)

churn_probability = 1 / (1 + np.exp(-logit))

will_churn = (
    rng.random(N_SUBSCRIPTIONS) < churn_probability
)


# ------------------------------------------------
# Subscription duration
# ------------------------------------------------

duration_days = np.clip(
    rng.exponential(scale=150, size=N_SUBSCRIPTIONS).astype(int) + 20,
    7,
    500
)

potential_end = (
    subscriptions["subscription_start_date"] +
    pd.to_timedelta(duration_days, unit="D")
)

# Підписка вважається cancelled тільки якщо її завершення
# реально потрапляє в наш observation period.
is_cancelled = (
    will_churn &
    (potential_end <= DATA_END)
)

subscriptions["subscription_end_date"] = pd.NaT

subscriptions.loc[
    is_cancelled,
    "subscription_end_date"
] = potential_end[is_cancelled]

subscriptions["status"] = np.where(
    is_cancelled,
    "cancelled",
    "active"
)


# ------------------------------------------------
# Cancellation reason
# ------------------------------------------------

reasons = [
    "too_expensive",
    "not_using_service",
    "poor_performance",
    "payment_issue",
    "missing_games",
    "other"
]

reason_prob = [
    0.26,
    0.24,
    0.17,
    0.11,
    0.12,
    0.10
]

subscriptions["cancellation_reason"] = None

cancelled_count = is_cancelled.sum()

subscriptions.loc[
    is_cancelled,
    "cancellation_reason"
] = rng.choice(
    reasons,
    size=cancelled_count,
    p=reason_prob
)


# ------------------------------------------------
# Залишаємо тільки реальні database columns
# ------------------------------------------------

subscriptions = subscriptions[[
    "subscription_id",
    "user_id",
    "plan",
    "subscription_start_date",
    "subscription_end_date",
    "status",
    "cancellation_reason"
]]


# Зберігаємо
subscriptions.to_csv(
    DATA_DIR / "subscriptions.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Rows:", len(subscriptions))
print("Unique subscribers:", subscriptions["user_id"].nunique())
print("\nStatus:")
print(subscriptions["status"].value_counts())

print("\nPlans:")
print(subscriptions["plan"].value_counts())

subscriptions.head(10)

Rows: 250000
Unique subscribers: 120000

Status:
status
active       158121
cancelled     91879
Name: count, dtype: int64

Plans:
plan
priority    179984
ultimate     70016
Name: count, dtype: int64


,subscription_id,user_id,plan,subscription_start_date,subscription_end_date,status,cancellation_reason
0,1,29673,priority,2024-04-10,2024-08-22,cancelled,payment_issue
1,2,12721,priority,2025-07-10,NaT,active,None
2,3,21374,ultimate,2026-04-26,NaT,active,None
3,4,60932,priority,2025-09-24,NaT,active,None
4,5,94293,ultimate,2026-08-31,NaT,active,None
5,6,60540,priority,2025-09-16,NaT,active,None
6,7,34837,priority,2025-03-27,2025-05-01,cancelled,not_using_service
7,8,33346,ultimate,2024-05-27,NaT,active,None
8,9,5411,ultimate,2024-03-25,2024-04-17,cancelled,not_using_service
9,10,65201,ultimate,2025-01-08,NaT,active,None


In [6]:
# ==========================================
# PAYMENTS — vectorized version
# ==========================================

# Копія subscriptions для генерації billing cycles
billing = subscriptions[[
    "subscription_id",
    "user_id",
    "plan",
    "subscription_start_date",
    "subscription_end_date"
]].copy()

billing["subscription_start_date"] = pd.to_datetime(
    billing["subscription_start_date"]
)

billing["subscription_end_date"] = pd.to_datetime(
    billing["subscription_end_date"]
)

DATA_END = pd.Timestamp("2026-08-31")

billing["observation_end"] = (
    billing["subscription_end_date"]
    .fillna(DATA_END)
)

# ------------------------------------------
# Скільки billing cycles має кожна підписка
# ------------------------------------------

billing["months_active"] = (
    (billing["observation_end"].dt.year -
     billing["subscription_start_date"].dt.year) * 12
    +
    (billing["observation_end"].dt.month -
     billing["subscription_start_date"].dt.month)
    + 1
)

billing["months_active"] = (
    billing["months_active"]
    .clip(lower=1)
)

# Створюємо по одному рядку на billing cycle
payments = billing.loc[
    billing.index.repeat(billing["months_active"])
].copy()

payments["billing_month"] = (
    payments.groupby("subscription_id")
    .cumcount()
)

print("Base billing cycles:", len(payments))

Base billing cycles: 2567980


In [7]:
# ------------------------------------------
# Payment dates
# ------------------------------------------

start_period = (
    payments["subscription_start_date"]
    .dt.to_period("M")
)

payment_period = (
    start_period +
    payments["billing_month"]
)

payments["payment_date"] = (
    payment_period.dt.to_timestamp()
    +
    pd.to_timedelta(
        payments["subscription_start_date"].dt.day - 1,
        unit="D"
    )
)

# Прибираємо платежі, які випадково вийшли
# за observation period
payments = payments[
    payments["payment_date"] <= payments["observation_end"]
].copy()


# ------------------------------------------
# Price
# ------------------------------------------

payments["amount"] = np.where(
    payments["plan"] == "priority",
    9.99,
    19.99
)

payments["currency"] = "USD"


# ------------------------------------------
# Payment method
# ------------------------------------------

payments["payment_method"] = rng.choice(
    ["card", "paypal", "apple_pay", "google_pay"],
    size=len(payments),
    p=[0.67, 0.16, 0.09, 0.08]
)


# ------------------------------------------
# Status
# ------------------------------------------

random_status = rng.random(len(payments))

payments["payment_status"] = np.select(
    [
        random_status < 0.055,
        random_status < 0.080
    ],
    [
        "failed",
        "refunded"
    ],
    default="successful"
)

payments["refund_amount"] = np.where(
    payments["payment_status"] == "refunded",
    payments["amount"],
    0.00
)


# ------------------------------------------
# Final table
# ------------------------------------------

payments = payments.reset_index(drop=True)

payments.insert(
    0,
    "payment_id",
    np.arange(1, len(payments) + 1)
)

payments = payments[[
    "payment_id",
    "subscription_id",
    "user_id",
    "payment_date",
    "amount",
    "currency",
    "payment_status",
    "payment_method",
    "refund_amount"
]]

print("Payment attempts:", f"{len(payments):,}")

print("\nStatuses:")
print(payments["payment_status"].value_counts())

payments.head(10)

Payment attempts: 2,523,131

Statuses:
payment_status
successful    2321175
failed         138551
refunded        63405
Name: count, dtype: int64


,payment_id,subscription_id,user_id,payment_date,amount,currency,payment_status,payment_method,refund_amount
0,1,1,29673,2024-04-10,9.99,USD,successful,paypal,0.0
1,2,1,29673,2024-05-10,9.99,USD,successful,google_pay,0.0
2,3,1,29673,2024-06-10,9.99,USD,successful,paypal,0.0
3,4,1,29673,2024-07-10,9.99,USD,successful,card,0.0
4,5,1,29673,2024-08-10,9.99,USD,successful,paypal,0.0
5,6,2,12721,2025-07-10,9.99,USD,successful,card,0.0
6,7,2,12721,2025-08-10,9.99,USD,successful,card,0.0
7,8,2,12721,2025-09-10,9.99,USD,successful,apple_pay,0.0
8,9,2,12721,2025-10-10,9.99,USD,successful,card,0.0
9,10,2,12721,2025-11-10,9.99,USD,successful,card,0.0


In [8]:
# ==========================================
# IMPROVE PAYMENT HISTORY
# ==========================================

# 1. Основний payment method для кожної subscription
subscription_methods = pd.DataFrame({
    "subscription_id": subscriptions["subscription_id"],
    "payment_method": rng.choice(
        ["card", "paypal", "apple_pay", "google_pay"],
        size=len(subscriptions),
        p=[0.67, 0.16, 0.09, 0.08]
    )
})

payments = payments.drop(columns="payment_method").merge(
    subscription_methods,
    on="subscription_id",
    how="left"
)


# 2. Створюємо retries для частини failed payments
failed = payments[
    payments["payment_status"] == "failed"
].copy()

retry_mask = rng.random(len(failed)) < 0.72

retries = failed[retry_mask].copy()

# Retry через 1–5 днів
retries["payment_date"] = (
    retries["payment_date"]
    + pd.to_timedelta(
        rng.integers(1, 6, len(retries)),
        unit="D"
    )
)

# Більшість retry успішні
retry_success = rng.random(len(retries)) < 0.86

retries["payment_status"] = np.where(
    retry_success,
    "successful",
    "failed"
)

retries["refund_amount"] = 0.00


# Не дозволяємо retry виходити за dataset period
retries = retries[
    retries["payment_date"] <= DATA_END
].copy()


# 3. Додаємо retries назад
payments = pd.concat(
    [payments, retries],
    ignore_index=True
)

# Сортуємо як справжню transaction history
payments = payments.sort_values(
    ["payment_date", "subscription_id"]
).reset_index(drop=True)

# Перегенеровуємо PK
payments["payment_id"] = np.arange(
    1,
    len(payments) + 1
)

# Порядок колонок
payments = payments[[
    "payment_id",
    "subscription_id",
    "user_id",
    "payment_date",
    "amount",
    "currency",
    "payment_status",
    "payment_method",
    "refund_amount"
]]

print("Final payment rows:", f"{len(payments):,}")

print("\nPayment statuses:")
print(payments["payment_status"].value_counts())

payments.head(10)

Final payment rows: 2,622,107

Payment statuses:
payment_status
successful    2406309
failed         152393
refunded        63405
Name: count, dtype: int64


,payment_id,subscription_id,user_id,payment_date,amount,currency,payment_status,payment_method,refund_amount
0,1,51385,83033,2024-01-01,19.99,USD,successful,card,0.0
1,2,127337,122816,2024-01-01,9.99,USD,successful,card,0.0
2,3,137182,40189,2024-01-01,19.99,USD,successful,card,0.0
3,4,150554,54141,2024-01-01,9.99,USD,successful,card,0.0
4,5,201219,109530,2024-01-01,9.99,USD,successful,card,0.0
5,6,32744,94414,2024-01-02,9.99,USD,successful,card,0.0
6,7,56990,56714,2024-01-02,9.99,USD,failed,apple_pay,0.0
7,8,85257,14062,2024-01-02,19.99,USD,successful,card,0.0
8,9,103807,98082,2024-01-02,9.99,USD,successful,card,0.0
9,10,149435,123360,2024-01-02,9.99,USD,successful,card,0.0


In [9]:
# ==========================================
# PAYMENT DATA QUALITY CHECK
# ==========================================

print("Rows:", f"{len(payments):,}")

print("\nDuplicate payment_id:")
print(payments["payment_id"].duplicated().sum())

print("\nMissing values:")
print(payments.isna().sum())

print("\nPayment date range:")
print(
    payments["payment_date"].min(),
    "→",
    payments["payment_date"].max()
)

print("\nInvalid amounts:")
print((payments["amount"] <= 0).sum())

print("\nRefund logic errors:")
print(
    (
        (payments["payment_status"] != "refunded") &
        (payments["refund_amount"] > 0)
    ).sum()
)

print("\nUnknown subscription IDs:")
print(
    (~payments["subscription_id"]
      .isin(subscriptions["subscription_id"]))
    .sum()
)

print("\nUnknown user IDs:")
print(
    (~payments["user_id"]
      .isin(users["user_id"]))
    .sum()
)

print("\nPayment status distribution:")
print(
    payments["payment_status"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Rows: 2,622,107

Duplicate payment_id:
0

Missing values:
payment_id         0
subscription_id    0
user_id            0
payment_date       0
amount             0
currency           0
payment_status     0
payment_method     0
refund_amount      0
dtype: int64

Payment date range:
2024-01-01 00:00:00 → 2026-08-31 00:00:00

Invalid amounts:
0

Refund logic errors:
0

Unknown subscription IDs:
0

Unknown user IDs:
0

Payment status distribution:
payment_status
successful    91.77
failed         5.81
refunded       2.42
Name: proportion, dtype: float64


In [10]:
payments.to_csv(
    DATA_DIR / "payments.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved:", DATA_DIR / "payments.csv")
print("Rows:", f"{len(payments):,}")

Saved: ..\data\payments.csv
Rows: 2,622,107


In [11]:
# ==========================================
# GAMING SESSIONS — 4,000,000 rows
# Chunk-based generation
# ==========================================

N_SESSIONS = 4_000_000
CHUNK_SIZE = 250_000

SESSIONS_FILE = DATA_DIR / "gaming_sessions.csv"

# Якщо файл уже існує після старої/невдалої генерації —
# видаляємо його перед новим запуском
if SESSIONS_FILE.exists():
    SESSIONS_FILE.unlink()

# -----------------------------
# User lookup arrays
# -----------------------------

user_ids = users["user_id"].to_numpy()
signup_dates = pd.to_datetime(users["signup_date"]).to_numpy()

user_devices = users["signup_device"].to_numpy()
user_countries = users["country"].to_numpy()

# hidden characteristics from generator
engagement_lookup = (
    user_profile
    .set_index("user_id")["engagement_propensity"]
    .reindex(user_ids)
    .to_numpy()
)

churn_lookup = (
    user_profile
    .set_index("user_id")["churn_propensity"]
    .reindex(user_ids)
    .to_numpy()
)

# -----------------------------
# Configuration
# -----------------------------

DATA_END = np.datetime64("2026-08-31")

server_regions = np.array([
    "us_east",
    "us_west",
    "eu_central",
    "eu_west",
    "asia_east",
    "asia_se",
    "sa_east"
])

# Country → most likely server region
country_region = {
    "US": "us_east",
    "CA": "us_east",

    "DE": "eu_central",
    "PL": "eu_central",
    "UA": "eu_central",
    "GE": "eu_central",
    "SE": "eu_central",
    "NL": "eu_west",
    "GB": "eu_west",
    "FR": "eu_west",
    "ES": "eu_west",
    "IT": "eu_central",

    "JP": "asia_east",
    "KR": "asia_east",

    "BR": "sa_east"
}

# 500 games
game_ids = np.arange(1, 501)

# Popularity is intentionally unequal
game_weights = 1 / np.power(
    np.arange(1, 501),
    0.85
)

game_weights = game_weights / game_weights.sum()


# ==========================================
# Generate chunks
# ==========================================

generated = 0

for chunk_start in range(0, N_SESSIONS, CHUNK_SIZE):

    n = min(
        CHUNK_SIZE,
        N_SESSIONS - chunk_start
    )

    # --------------------------------------
    # USERS
    # --------------------------------------

    # Higher-engagement users should appear more often.
    # Convert latent engagement to sampling weights.
    weights = np.exp(
        np.clip(engagement_lookup, -2, 2) * 0.55
    )

    weights = weights / weights.sum()

    selected_idx = rng.choice(
        len(user_ids),
        size=n,
        replace=True,
        p=weights
    )

    selected_users = user_ids[selected_idx]
    selected_signup = signup_dates[selected_idx]
    selected_devices = user_devices[selected_idx]
    selected_countries = user_countries[selected_idx]

    selected_engagement = engagement_lookup[selected_idx]
    selected_churn = churn_lookup[selected_idx]


    # --------------------------------------
    # SESSION DATE
    # --------------------------------------

    days_available = (
        DATA_END -
        selected_signup.astype("datetime64[D]")
    ).astype(int)

    days_available = np.maximum(
        days_available,
        0
    )

    random_fraction = rng.random(n)

    days_after_signup = (
        random_fraction *
        (days_available + 1)
    ).astype(int)

    session_dates = (
        selected_signup.astype("datetime64[D]") +
        days_after_signup.astype("timedelta64[D]")
    )


    # --------------------------------------
    # TIME OF DAY
    # --------------------------------------

    hours = rng.choice(
        np.arange(24),
        size=n,
        p=np.array([
            0.015, 0.010, 0.008, 0.007,
            0.007, 0.010, 0.015, 0.020,
            0.025, 0.030, 0.035, 0.040,
            0.045, 0.045, 0.050, 0.055,
            0.065, 0.075, 0.090, 0.100,
            0.095, 0.075, 0.050, 0.028
        ]) / 0.995
    )

    minutes = rng.integers(0, 60, n)
    seconds = rng.integers(0, 60, n)

    session_start = (
        session_dates.astype("datetime64[s]")
        + hours.astype("timedelta64[h]")
        + minutes.astype("timedelta64[m]")
        + seconds.astype("timedelta64[s]")
    )


    # --------------------------------------
    # SESSION DURATION
    # --------------------------------------

    duration_minutes = np.clip(
        rng.lognormal(
            mean=4.15 + 0.08 * selected_engagement,
            sigma=0.70,
            size=n
        ),
        2,
        420
    ).astype(int)

    session_end = (
        session_start +
        duration_minutes.astype("timedelta64[m]")
    )


    # --------------------------------------
    # GAME
    # --------------------------------------

    selected_games = rng.choice(
        game_ids,
        size=n,
        p=game_weights
    )


    # --------------------------------------
    # SERVER REGION
    # --------------------------------------

    preferred_regions = np.array([
        country_region.get(country, "eu_central")
        for country in selected_countries
    ])

    # ~90% connect to expected region;
    # some users get routed elsewhere.
    wrong_region = rng.random(n) < 0.10

    selected_regions = preferred_regions.copy()

    selected_regions[wrong_region] = rng.choice(
        server_regions,
        size=wrong_region.sum()
    )


    # --------------------------------------
    # QUEUE TIME
    # --------------------------------------

    peak_hours = (
        (hours >= 18) &
        (hours <= 22)
    )

    queue_time = rng.normal(
        loc=35
            + peak_hours * 85
            + np.maximum(selected_churn, 0) * 12,
        scale=35,
        size=n
    )

    queue_time = np.clip(
        queue_time,
        0,
        900
    ).astype(int)


    # --------------------------------------
    # LATENCY
    # --------------------------------------

    wrong_region_penalty = (
        selected_regions != preferred_regions
    ) * 45

    country_penalty = np.isin(
        selected_countries,
        ["BR", "GE"]
    ) * 10

    latency = rng.normal(
        loc=38
            + wrong_region_penalty
            + country_penalty,
        scale=14,
        size=n
    )

    latency = np.clip(
        latency,
        10,
        250
    ).astype(int)


    # --------------------------------------
    # FPS
    # --------------------------------------

    fps = rng.normal(
        loc=78 - np.maximum(latency - 45, 0) * 0.18,
        scale=11,
        size=n
    )

    fps = np.clip(
        fps,
        20,
        120
    ).astype(int)


    # --------------------------------------
    # DISCONNECT
    # --------------------------------------

    disconnect_probability = (
        0.012
        + latency / 1500
        + queue_time / 10000
    )

    disconnect_probability = np.clip(
        disconnect_probability,
        0.01,
        0.25
    )

    disconnect_flag = (
        rng.random(n) <
        disconnect_probability
    ).astype(int)


    # --------------------------------------
    # CREATE CHUNK
    # --------------------------------------

    session_ids = np.arange(
        chunk_start + 1,
        chunk_start + n + 1
    )

    chunk = pd.DataFrame({
        "session_id": session_ids,
        "user_id": selected_users,
        "game_id": selected_games,
        "session_start": session_start,
        "session_end": session_end,
        "device": selected_devices,
        "server_region": selected_regions,
        "queue_time_seconds": queue_time,
        "avg_latency_ms": latency,
        "avg_fps": fps,
        "disconnect_flag": disconnect_flag
    })


    # --------------------------------------
    # WRITE TO CSV
    # --------------------------------------

    chunk.to_csv(
        SESSIONS_FILE,
        mode="a",
        header=(chunk_start == 0),
        index=False
    )

    generated += n

    print(
        f"Generated: {generated:,} / {N_SESSIONS:,}"
    )


print("\nDONE")
print("gaming_sessions.csv created")

Generated: 250,000 / 4,000,000
Generated: 500,000 / 4,000,000
Generated: 750,000 / 4,000,000
Generated: 1,000,000 / 4,000,000
Generated: 1,250,000 / 4,000,000
Generated: 1,500,000 / 4,000,000
Generated: 1,750,000 / 4,000,000
Generated: 2,000,000 / 4,000,000
Generated: 2,250,000 / 4,000,000
Generated: 2,500,000 / 4,000,000
Generated: 2,750,000 / 4,000,000
Generated: 3,000,000 / 4,000,000
Generated: 3,250,000 / 4,000,000
Generated: 3,500,000 / 4,000,000
Generated: 3,750,000 / 4,000,000
Generated: 4,000,000 / 4,000,000

DONE
gaming_sessions.csv created


In [12]:
# ==========================================
# GAMING SESSIONS — QUICK QA
# ==========================================

# Читаємо тільки невелику вибірку
sessions_sample = pd.read_csv(
    DATA_DIR / "gaming_sessions.csv",
    nrows=100_000,
    parse_dates=["session_start", "session_end"]
)

print("Sample shape:", sessions_sample.shape)

print("\nMissing values:")
print(sessions_sample.isna().sum())

print("\nInvalid session duration:")
print(
    (sessions_sample["session_end"] <=
     sessions_sample["session_start"]).sum()
)

print("\nLatency:")
print(sessions_sample["avg_latency_ms"].describe())

print("\nQueue time:")
print(sessions_sample["queue_time_seconds"].describe())

print("\nFPS:")
print(sessions_sample["avg_fps"].describe())

print("\nDisconnect rate:")
print(
    round(
        sessions_sample["disconnect_flag"].mean() * 100,
        2
    ),
    "%"
)

print("\nServer regions:")
print(
    sessions_sample["server_region"]
    .value_counts()
)

Sample shape: (100000, 11)

Missing values:
session_id            0
user_id               0
game_id               0
session_start         0
session_end           0
device                0
server_region         0
queue_time_seconds    0
avg_latency_ms        0
avg_fps               0
disconnect_flag       0
dtype: int64

Invalid session duration:
0

Latency:
count    100000.000000
mean         42.429840
std          18.854069
min          10.000000
25%          30.000000
50%          40.000000
75%          52.000000
max         142.000000
Name: avg_latency_ms, dtype: float64

Queue time:
count    100000.000000
mean         75.766440
std          52.664676
min           0.000000
25%          32.000000
50%          70.000000
75%         117.000000
max         277.000000
Name: queue_time_seconds, dtype: float64

FPS:
count    100000.00000
mean         76.42083
std          11.26573
min          21.00000
25%          69.00000
50%          76.00000
75%          84.00000
max         120.00000

In [13]:
# ==========================================
# EVENTS — PART 1
# Gaming journeys from gaming_sessions.csv
# ==========================================

SESSIONS_FILE = DATA_DIR / "gaming_sessions.csv"
EVENTS_FILE = DATA_DIR / "events.csv"

READ_CHUNK = 250_000

# Start from scratch
if EVENTS_FILE.exists():
    EVENTS_FILE.unlink()

event_id_counter = 1
total_events = 0

# ------------------------------------------
# Read gaming sessions in chunks
# ------------------------------------------

for sessions in pd.read_csv(
    SESSIONS_FILE,
    chunksize=READ_CHUNK,
    parse_dates=["session_start", "session_end"]
):

    n = len(sessions)

    # ======================================
    # 1. GAME LAUNCH
    # ======================================
    # Happens shortly before session_start

    launch_seconds_before = rng.integers(
        20,
        181,
        size=n
    )

    launch_time = (
        sessions["session_start"]
        - pd.to_timedelta(
            launch_seconds_before,
            unit="s"
        )
    )

    launch_events = pd.DataFrame({
        "user_id": sessions["user_id"].values,
        "event_timestamp": launch_time.values,
        "event_type": "game_launch",
        "gaming_session_id": sessions["session_id"].values,
        "game_id": sessions["game_id"].values,
        "device": sessions["device"].values
    })

    # ======================================
    # 2. GAME STARTED
    # ======================================

    start_events = pd.DataFrame({
        "user_id": sessions["user_id"].values,
        "event_timestamp": sessions["session_start"].values,
        "event_type": "game_started",
        "gaming_session_id": sessions["session_id"].values,
        "game_id": sessions["game_id"].values,
        "device": sessions["device"].values
    })

    # ======================================
    # 3. GAME ENDED
    # ======================================
    #
    # Only normal sessions.
    # Disconnect sessions instead receive
    # disconnect event.
    # ======================================

    normal_mask = (
        sessions["disconnect_flag"] == 0
    )

    normal_sessions = sessions.loc[
        normal_mask
    ]

    end_events = pd.DataFrame({
        "user_id":
            normal_sessions["user_id"].values,

        "event_timestamp":
            normal_sessions["session_end"].values,

        "event_type":
            "game_ended",

        "gaming_session_id":
            normal_sessions["session_id"].values,

        "game_id":
            normal_sessions["game_id"].values,

        "device":
            normal_sessions["device"].values
    })

    # ======================================
    # 4. DISCONNECT
    # ======================================

    disconnect_sessions = sessions.loc[
        sessions["disconnect_flag"] == 1
    ].copy()

    # Disconnect somewhere between
    # session_start and session_end

    duration_seconds = (
        disconnect_sessions["session_end"]
        - disconnect_sessions["session_start"]
    ).dt.total_seconds().astype(int)

    # Avoid edge cases
    duration_seconds = np.maximum(
        duration_seconds,
        2
    )

    # Usually not immediately after launch.
    # Position between 10% and 95% of session.
    disconnect_fraction = rng.uniform(
        0.10,
        0.95,
        size=len(disconnect_sessions)
    )

    disconnect_offset = (
        duration_seconds *
        disconnect_fraction
    ).astype(int)

    disconnect_time = (
        disconnect_sessions["session_start"]
        + pd.to_timedelta(
            disconnect_offset,
            unit="s"
        )
    )

    disconnect_events = pd.DataFrame({
        "user_id":
            disconnect_sessions["user_id"].values,

        "event_timestamp":
            disconnect_time.values,

        "event_type":
            "disconnect",

        "gaming_session_id":
            disconnect_sessions["session_id"].values,

        "game_id":
            disconnect_sessions["game_id"].values,

        "device":
            disconnect_sessions["device"].values
    })

    # ======================================
    # COMBINE CHUNK
    # ======================================

    events_chunk = pd.concat(
        [
            launch_events,
            start_events,
            end_events,
            disconnect_events
        ],
        ignore_index=True
    )

    # Sort chronologically inside chunk
    events_chunk = (
        events_chunk
        .sort_values("event_timestamp")
        .reset_index(drop=True)
    )

    # ======================================
    # EVENT ID
    # ======================================

    chunk_event_count = len(events_chunk)

    events_chunk.insert(
        0,
        "event_id",
        np.arange(
            event_id_counter,
            event_id_counter + chunk_event_count
        )
    )

    event_id_counter += chunk_event_count
    total_events += chunk_event_count

    # ======================================
    # WRITE
    # ======================================

    events_chunk.to_csv(
        EVENTS_FILE,
        mode="a",
        header=(total_events == chunk_event_count),
        index=False,
        date_format="%Y-%m-%d %H:%M:%S"
    )

    print(
        f"Sessions processed: "
        f"{sessions['session_id'].iloc[-1]:,} | "
        f"Events: {total_events:,}"
    )


print("\nDONE")
print(f"Gaming events created: {total_events:,}")
print("File:", EVENTS_FILE)

Sessions processed: 250,000 | Events: 750,000
Sessions processed: 500,000 | Events: 1,500,000
Sessions processed: 750,000 | Events: 2,250,000
Sessions processed: 1,000,000 | Events: 3,000,000
Sessions processed: 1,250,000 | Events: 3,750,000
Sessions processed: 1,500,000 | Events: 4,500,000
Sessions processed: 1,750,000 | Events: 5,250,000
Sessions processed: 2,000,000 | Events: 6,000,000
Sessions processed: 2,250,000 | Events: 6,750,000
Sessions processed: 2,500,000 | Events: 7,500,000
Sessions processed: 2,750,000 | Events: 8,250,000
Sessions processed: 3,000,000 | Events: 9,000,000
Sessions processed: 3,250,000 | Events: 9,750,000
Sessions processed: 3,500,000 | Events: 10,500,000
Sessions processed: 3,750,000 | Events: 11,250,000
Sessions processed: 4,000,000 | Events: 12,000,000

DONE
Gaming events created: 12,000,000
File: ..\data\events.csv


In [14]:
# ==========================================
# EVENTS <-> GAMING SESSIONS QA
# ==========================================

events_sample = pd.read_csv(
    DATA_DIR / "events.csv",
    nrows=750_000,
    parse_dates=["event_timestamp"]
)

sessions_sample = pd.read_csv(
    DATA_DIR / "gaming_sessions.csv",
    nrows=250_000,
    parse_dates=["session_start", "session_end"]
)

print("Events sample:", len(events_sample))
print("Sessions sample:", len(sessions_sample))

# ------------------------------------------
# Basic checks
# ------------------------------------------

print("\nMissing values:")
print(events_sample.isna().sum())

print("\nDuplicate event_id:")
print(events_sample["event_id"].duplicated().sum())

print("\nEvent types:")
print(events_sample["event_type"].value_counts())

# ------------------------------------------
# Every session should have exactly 3 events
# ------------------------------------------

events_per_session = (
    events_sample
    .groupby("gaming_session_id")
    .size()
)

print("\nEvents per gaming session:")
print(events_per_session.value_counts().sort_index())

# ------------------------------------------
# JOIN with gaming_sessions
# ------------------------------------------

qa = events_sample.merge(
    sessions_sample[
        [
            "session_id",
            "user_id",
            "game_id",
            "session_start",
            "session_end",
            "disconnect_flag"
        ]
    ],
    left_on="gaming_session_id",
    right_on="session_id",
    how="left",
    suffixes=("_event", "_session")
)

print("\nUnknown gaming_session_id:")
print(qa["session_id"].isna().sum())

print("\nUser mismatch:")
print(
    (
        qa["user_id_event"] !=
        qa["user_id_session"]
    ).sum()
)

print("\nGame mismatch:")
print(
    (
        qa["game_id_event"] !=
        qa["game_id_session"]
    ).sum()
)

# ------------------------------------------
# Event logic
# ------------------------------------------

launch_errors = (
    (qa["event_type"] == "game_launch") &
    (qa["event_timestamp"] >= qa["session_start"])
).sum()

start_errors = (
    (qa["event_type"] == "game_started") &
    (qa["event_timestamp"] != qa["session_start"])
).sum()

end_errors = (
    (qa["event_type"] == "game_ended") &
    (qa["event_timestamp"] != qa["session_end"])
).sum()

disconnect_errors = (
    (qa["event_type"] == "disconnect") &
    (
        (qa["event_timestamp"] <= qa["session_start"]) |
        (qa["event_timestamp"] >= qa["session_end"])
    )
).sum()

print("\nTimestamp logic errors:")
print("game_launch:", launch_errors)
print("game_started:", start_errors)
print("game_ended:", end_errors)
print("disconnect:", disconnect_errors)

# ------------------------------------------
# Disconnect consistency
# ------------------------------------------

wrong_disconnect = (
    (qa["event_type"] == "disconnect") &
    (qa["disconnect_flag"] != 1)
).sum()

wrong_normal_end = (
    (qa["event_type"] == "game_ended") &
    (qa["disconnect_flag"] != 0)
).sum()

print("\nDisconnect logic errors:")
print("disconnect event with flag != 1:", wrong_disconnect)
print("game_ended with flag != 0:", wrong_normal_end)

Events sample: 750000
Sessions sample: 250000

Missing values:
event_id             0
user_id              0
event_timestamp      0
event_type           0
gaming_session_id    0
game_id              0
device               0
dtype: int64

Duplicate event_id:
0

Event types:
event_type
game_launch     250000
game_started    250000
game_ended      237829
disconnect       12171
Name: count, dtype: int64

Events per gaming session:
3    250000
Name: count, dtype: int64

Unknown gaming_session_id:
0

User mismatch:
0

Game mismatch:
0

Timestamp logic errors:
game_launch: 0
game_started: 0
game_ended: 0
disconnect: 0

Disconnect logic errors:
disconnect event with flag != 1: 0
game_ended with flag != 0: 0


In [15]:
# ==========================================
# PRODUCT EVENTS — PART 2A
# Journeys leading to real gaming sessions
# ==========================================

SESSIONS_FILE = DATA_DIR / "gaming_sessions.csv"
PRODUCT_EVENTS_FILE = DATA_DIR / "product_events.csv"

READ_CHUNK = 250_000

if PRODUCT_EVENTS_FILE.exists():
    PRODUCT_EVENTS_FILE.unlink()

# IDs continue after existing 12M gaming events
event_id_counter = 12_000_001
total_product_events = 0

for sessions in pd.read_csv(
    SESSIONS_FILE,
    chunksize=READ_CHUNK,
    parse_dates=["session_start"]
):

    n = len(sessions)

    # --------------------------------------
    # Existing game_launch happened
    # 20–180 sec before session_start.
    #
    # We don't know the exact generated
    # launch timestamp here, so build
    # product journey safely before the
    # earliest possible launch.
    # --------------------------------------

    journey_anchor = (
        sessions["session_start"]
        - pd.to_timedelta(4, unit="m")
    )

    # ======================================
    # APP OPEN
    # ======================================

    app_open_time = (
        journey_anchor
        - pd.to_timedelta(
            rng.integers(30, 181, size=n),
            unit="s"
        )
    )

    app_open = pd.DataFrame({
        "user_id": sessions["user_id"].values,
        "event_timestamp": app_open_time.values,
        "event_type": "app_open",
        "gaming_session_id": pd.array(
            sessions["session_id"].values,
            dtype="Int64"
        ),
        "game_id": pd.array(
            [pd.NA] * n,
            dtype="Int64"
        ),
        "device": sessions["device"].values
    })

    # ======================================
    # LOGIN
    # ======================================
    # Most successful gaming journeys
    # contain login, but not all:
    # persistent authentication exists.

    login_mask = rng.random(n) < 0.82
    login_sessions = sessions.loc[login_mask]

    login_time = (
        journey_anchor.loc[login_mask]
        - pd.to_timedelta(
            rng.integers(
                15,
                90,
                size=login_mask.sum()
            ),
            unit="s"
        )
    )

    login = pd.DataFrame({
        "user_id":
            login_sessions["user_id"].values,

        "event_timestamp":
            login_time.values,

        "event_type":
            "login",

        "gaming_session_id":
            pd.array(
                login_sessions["session_id"].values,
                dtype="Int64"
            ),

        "game_id":
            pd.array(
                [pd.NA] * len(login_sessions),
                dtype="Int64"
            ),

        "device":
            login_sessions["device"].values
    })

    # ======================================
    # GAME SEARCH
    # ======================================
    # Many users launch from favourites /
    # recent games, therefore search isn't
    # required for every session.

    search_mask = rng.random(n) < 0.58
    search_sessions = sessions.loc[search_mask]

    search_time = (
        journey_anchor.loc[search_mask]
        + pd.to_timedelta(
            rng.integers(
                5,
                31,
                size=search_mask.sum()
            ),
            unit="s"
        )
    )

    game_search = pd.DataFrame({
        "user_id":
            search_sessions["user_id"].values,

        "event_timestamp":
            search_time.values,

        "event_type":
            "game_search",

        "gaming_session_id":
            pd.array(
                search_sessions["session_id"].values,
                dtype="Int64"
            ),

        "game_id":
            pd.array(
                search_sessions["game_id"].values,
                dtype="Int64"
            ),

        "device":
            search_sessions["device"].values
    })

    # ======================================
    # GAME VIEW
    # ======================================
    # Almost every launch has a preceding
    # game page/view, but some are direct.

    view_mask = rng.random(n) < 0.88
    view_sessions = sessions.loc[view_mask]

    view_time = (
        journey_anchor.loc[view_mask]
        + pd.to_timedelta(
            rng.integers(
                35,
                91,
                size=view_mask.sum()
            ),
            unit="s"
        )
    )

    game_view = pd.DataFrame({
        "user_id":
            view_sessions["user_id"].values,

        "event_timestamp":
            view_time.values,

        "event_type":
            "game_view",

        "gaming_session_id":
            pd.array(
                view_sessions["session_id"].values,
                dtype="Int64"
            ),

        "game_id":
            pd.array(
                view_sessions["game_id"].values,
                dtype="Int64"
            ),

        "device":
            view_sessions["device"].values
    })

    # ======================================
    # COMBINE
    # ======================================

    chunk = pd.concat(
        [
            app_open,
            login,
            game_search,
            game_view
        ],
        ignore_index=True
    )

    chunk = (
        chunk
        .sort_values("event_timestamp")
        .reset_index(drop=True)
    )

    count = len(chunk)

    chunk.insert(
        0,
        "event_id",
        np.arange(
            event_id_counter,
            event_id_counter + count
        )
    )

    event_id_counter += count
    total_product_events += count

    chunk.to_csv(
        PRODUCT_EVENTS_FILE,
        mode="a",
        header=(total_product_events == count),
        index=False,
        date_format="%Y-%m-%d %H:%M:%S"
    )

    print(
        f"Sessions processed: "
        f"{sessions['session_id'].iloc[-1]:,} | "
        f"Product events: "
        f"{total_product_events:,}"
    )


print("\nDONE")
print(
    "Product events created:",
    f"{total_product_events:,}"
)

Sessions processed: 250,000 | Product events: 819,696
Sessions processed: 500,000 | Product events: 1,640,189
Sessions processed: 750,000 | Product events: 2,460,339
Sessions processed: 1,000,000 | Product events: 3,279,555
Sessions processed: 1,250,000 | Product events: 4,099,327
Sessions processed: 1,500,000 | Product events: 4,919,479
Sessions processed: 1,750,000 | Product events: 5,739,768
Sessions processed: 2,000,000 | Product events: 6,559,679
Sessions processed: 2,250,000 | Product events: 7,379,795
Sessions processed: 2,500,000 | Product events: 8,200,207
Sessions processed: 2,750,000 | Product events: 9,019,790
Sessions processed: 3,000,000 | Product events: 9,839,724
Sessions processed: 3,250,000 | Product events: 10,659,579
Sessions processed: 3,500,000 | Product events: 11,479,549
Sessions processed: 3,750,000 | Product events: 12,299,529
Sessions processed: 4,000,000 | Product events: 13,119,778

DONE
Product events created: 13,119,778


In [16]:
# ==========================================
# PRODUCT EVENTS — QUICK QA
# ==========================================

product_sample = pd.read_csv(
    DATA_DIR / "product_events.csv",
    nrows=500_000,
    parse_dates=["event_timestamp"]
)

print("Sample rows:", len(product_sample))

print("\nEvent distribution:")
print(product_sample["event_type"].value_counts())

print("\nMissing values:")
print(product_sample.isna().sum())

print("\nDuplicate event_id:")
print(product_sample["event_id"].duplicated().sum())

print("\nEvent ID range:")
print(
    product_sample["event_id"].min(),
    "→",
    product_sample["event_id"].max()
)

print("\nUnknown users:")
print(
    (~product_sample["user_id"]
     .isin(users["user_id"]))
    .sum()
)

print("\nInvalid gaming_session_id:")
print(
    (
        (product_sample["gaming_session_id"] < 1) |
        (product_sample["gaming_session_id"] > 4_000_000)
    ).sum()
)

print("\nEvent types:")
print(
    sorted(
        product_sample["event_type"].unique()
    )
)

Sample rows: 500000

Event distribution:
event_type
app_open       152503
game_view      134304
login          125158
game_search     88035
Name: count, dtype: int64

Missing values:
event_id                  0
user_id                   0
event_timestamp           0
event_type                0
gaming_session_id         0
game_id              277661
device                    0
dtype: int64

Duplicate event_id:
0

Event ID range:
12000001 → 12500000

Unknown users:
0

Invalid gaming_session_id:
0

Event types:
['app_open', 'game_search', 'game_view', 'login']


In [17]:
# ==========================================
# ADD journey_id TO SUCCESSFUL PRODUCT EVENTS
# Chunk-based rewrite
# ==========================================

PRODUCT_EVENTS_FILE = DATA_DIR / "product_events.csv"
TEMP_FILE = DATA_DIR / "product_events_with_journey.csv"

CHUNK_SIZE = 500_000

if TEMP_FILE.exists():
    TEMP_FILE.unlink()

processed = 0

for chunk in pd.read_csv(
    PRODUCT_EVENTS_FILE,
    chunksize=CHUNK_SIZE
):

    # Successful journey corresponds
    # to an existing gaming session
    chunk["journey_id"] = (
        chunk["gaming_session_id"]
        .astype("Int64")
    )

    # Put journey_id in a logical position
    column_order = [
        "event_id",
        "user_id",
        "event_timestamp",
        "event_type",
        "journey_id",
        "gaming_session_id",
        "game_id",
        "device"
    ]

    chunk = chunk[column_order]

    chunk.to_csv(
        TEMP_FILE,
        mode="a",
        header=(processed == 0),
        index=False
    )

    processed += len(chunk)

    print(
        f"Processed: {processed:,}"
    )

print("\nDONE")
print("Rows:", f"{processed:,}")
print("File:", TEMP_FILE)

Processed: 500,000
Processed: 1,000,000
Processed: 1,500,000
Processed: 2,000,000
Processed: 2,500,000
Processed: 3,000,000
Processed: 3,500,000
Processed: 4,000,000
Processed: 4,500,000
Processed: 5,000,000
Processed: 5,500,000
Processed: 6,000,000
Processed: 6,500,000
Processed: 7,000,000
Processed: 7,500,000
Processed: 8,000,000
Processed: 8,500,000
Processed: 9,000,000
Processed: 9,500,000
Processed: 10,000,000
Processed: 10,500,000
Processed: 11,000,000
Processed: 11,500,000
Processed: 12,000,000
Processed: 12,500,000
Processed: 13,000,000
Processed: 13,119,778

DONE
Rows: 13,119,778
File: ..\data\product_events_with_journey.csv


In [18]:
# ==========================================
# QA — PRODUCT EVENTS WITH JOURNEY_ID
# ==========================================

FILE = DATA_DIR / "product_events_with_journey.csv"

sample = pd.read_csv(FILE, nrows=500_000)

print("Sample rows:", f"{len(sample):,}")

print("\nMissing values:")
print(sample.isna().sum())

print("\nDuplicate event_id:")
print(sample["event_id"].duplicated().sum())

print("\nJourney ID range:")
print(
    int(sample["journey_id"].min()),
    "->",
    int(sample["journey_id"].max())
)

# journey_id and gaming_session_id must match
mismatch = (
    sample["journey_id"].astype("Int64")
    != sample["gaming_session_id"].astype("Int64")
).sum()

print("\nJourney / gaming session mismatch:")
print(mismatch)

# One journey must belong to exactly one user
journey_users = (
    sample.groupby("journey_id")["user_id"]
    .nunique()
)

print("\nJourneys linked to >1 user:")
print((journey_users > 1).sum())

print("\nEvent types:")
print(sample["event_type"].value_counts())

sample.head(10)

Sample rows: 500,000

Missing values:
event_id                  0
user_id                   0
event_timestamp           0
event_type                0
journey_id                0
gaming_session_id         0
game_id              277661
device                    0
dtype: int64

Duplicate event_id:
0

Journey ID range:
2 -> 250000

Journey / gaming session mismatch:
0

Journeys linked to >1 user:
0

Event types:
event_type
app_open       152503
game_view      134304
login          125158
game_search     88035
Name: count, dtype: int64


,event_id,user_id,event_timestamp,event_type,journey_id,gaming_session_id,game_id,device
0,12000001,121236,2024-01-01 09:56:26,app_open,138530,138530,NaN,ios
1,12000002,121236,2024-01-01 09:59:03,login,138530,138530,NaN,ios
2,12000003,121236,2024-01-01 09:59:39,game_search,138530,138530,17.0,ios
3,12000004,28514,2024-01-04 14:26:09,app_open,47881,47881,NaN,browser
4,12000005,28514,2024-01-04 14:27:09,game_search,47881,47881,136.0,browser
5,12000006,28514,2024-01-04 14:27:45,game_view,47881,47881,136.0,browser
6,12000007,91500,2024-01-05 14:40:06,app_open,184415,184415,NaN,macos
7,12000008,91500,2024-01-05 14:41:19,login,184415,184415,NaN,macos
8,12000009,91500,2024-01-05 14:42:16,game_view,184415,184415,98.0,macos
9,12000010,45158,2024-01-05 15:25:49,app_open,77227,77227,NaN,android


In [22]:
# ============================================================
# GENERATE ABANDONED PRODUCT JOURNEYS
# ============================================================

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

ABANDONED_FILE = DATA_DIR / "abandoned_product_events.csv"

N_JOURNEYS = 1_200_000
CHUNK_JOURNEYS = 100_000

# Successful journeys use gaming_session_id as journey_id.
# We start abandoned journeys AFTER the highest session ID.
MAX_SUCCESSFUL_JOURNEY_ID = int(sessions["session_id"].max())

next_journey_id = MAX_SUCCESSFUL_JOURNEY_ID + 1

# Event IDs must continue after existing product events.
# We already know product events end around 13.1M,
# but calculate it explicitly.
PRODUCT_FILE = DATA_DIR / "product_events_with_journey.csv"

max_event_id = 0

for chunk in pd.read_csv(
    PRODUCT_FILE,
    usecols=["event_id"],
    chunksize=500_000
):
    max_event_id = max(
        max_event_id,
        int(chunk["event_id"].max())
    )

next_event_id = max_event_id + 1

print("Starting journey_id:", f"{next_journey_id:,}")
print("Starting event_id:", f"{next_event_id:,}")

# ------------------------------------------------------------
# User pool
# ------------------------------------------------------------

user_ids = users["user_id"].to_numpy()

# Product devices
devices = np.array([
    "windows",
    "macos",
    "android",
    "ios",
    "browser",
    "smart_tv"
])

device_probs = np.array([
    0.40,
    0.08,
    0.20,
    0.12,
    0.15,
    0.05
])

# Games
#game_ids = games["game_id"].to_numpy()

# ------------------------------------------------------------
# Abandonment stage
#
# 1 = app_open only
# 2 = app_open -> login
# 3 = ... -> game_search
# 4 = ... -> game_view
# ------------------------------------------------------------

stage_values = np.array([1, 2, 3, 4])

stage_probs = np.array([
    0.12,
    0.18,
    0.25,
    0.45
])

event_sequence = [
    "app_open",
    "login",
    "game_search",
    "game_view"
]

# ------------------------------------------------------------
# Generate in chunks
# ------------------------------------------------------------

if ABANDONED_FILE.exists():
    ABANDONED_FILE.unlink()

total_events = 0
total_journeys = 0

for start in range(0, N_JOURNEYS, CHUNK_JOURNEYS):

    n = min(
        CHUNK_JOURNEYS,
        N_JOURNEYS - start
    )

    journey_ids = np.arange(
        next_journey_id,
        next_journey_id + n
    )

    next_journey_id += n

    selected_users = rng.choice(
        user_ids,
        size=n,
        replace=True
    )

    selected_devices = rng.choice(
        devices,
        size=n,
        p=device_probs
    )

    selected_games = rng.choice(
        game_ids,
        size=n,
        replace=True
    )

    stages = rng.choice(
        stage_values,
        size=n,
        p=stage_probs
    )

    # Random journey start between project dates
    start_dates = pd.Timestamp("2024-01-01")

    total_seconds = int(
        (
            pd.Timestamp("2026-08-31 23:00:00")
            - start_dates
        ).total_seconds()
    )

    random_seconds = rng.integers(
        0,
        total_seconds,
        size=n
    )

    journey_starts = (
        start_dates
        + pd.to_timedelta(random_seconds, unit="s")
    )

    rows = []

    for stage_number, event_name in enumerate(
        event_sequence,
        start=1
    ):

        mask = stages >= stage_number

        count = mask.sum()

        if count == 0:
            continue

        # Each next funnel step occurs later.
        # app_open = start
        # login ~ seconds later
        # search ~ later
        # view ~ later

        if stage_number == 1:
            delay = np.zeros(count, dtype=int)

        elif stage_number == 2:
            delay = rng.integers(
                5,
                90,
                size=count
            )

        elif stage_number == 3:
            delay = rng.integers(
                30,
                240,
                size=count
            )

        else:
            delay = rng.integers(
                90,
                480,
                size=count
            )

        timestamps = (
            journey_starts[mask]
            + pd.to_timedelta(delay, unit="s")
        )

        # game_id only becomes relevant at search/view
        if stage_number >= 3:
            event_game_ids = selected_games[mask]
        else:
            event_game_ids = np.full(
                count,
                np.nan
            )

        event_df = pd.DataFrame({
            "user_id": selected_users[mask],
            "event_timestamp": timestamps,
            "event_type": event_name,
            "journey_id": journey_ids[mask],

            # Important:
            # abandoned journeys have NO gaming session
            "gaming_session_id": pd.Series(
                [pd.NA] * count,
                dtype="Int64"
            ),

            "game_id": event_game_ids,
            "device": selected_devices[mask]
        })

        rows.append(event_df)

    chunk_events = pd.concat(
        rows,
        ignore_index=True
    )

    # Sort journey events chronologically
    chunk_events = chunk_events.sort_values(
        ["journey_id", "event_timestamp"]
    ).reset_index(drop=True)

    # Assign unique event IDs
    chunk_events.insert(
        0,
        "event_id",
        np.arange(
            next_event_id,
            next_event_id + len(chunk_events)
        )
    )

    next_event_id += len(chunk_events)

    chunk_events.to_csv(
        ABANDONED_FILE,
        mode="a",
        header=(total_events == 0),
        index=False
    )

    total_events += len(chunk_events)
    total_journeys += n

    print(
        f"Journeys: {total_journeys:,} | "
        f"Events: {total_events:,}"
    )

print("\nDONE")
print(
    "Abandoned journeys:",
    f"{total_journeys:,}"
)
print(
    "Product events created:",
    f"{total_events:,}"
)
print(
    "File:",
    ABANDONED_FILE
)

Starting journey_id: 4,000,001
Starting event_id: 25,119,779
Journeys: 100,000 | Events: 303,200
Journeys: 200,000 | Events: 606,135
Journeys: 300,000 | Events: 909,518
Journeys: 400,000 | Events: 1,212,361
Journeys: 500,000 | Events: 1,515,605
Journeys: 600,000 | Events: 1,818,990
Journeys: 700,000 | Events: 2,121,619
Journeys: 800,000 | Events: 2,424,612
Journeys: 900,000 | Events: 2,727,562
Journeys: 1,000,000 | Events: 3,030,732
Journeys: 1,100,000 | Events: 3,333,704
Journeys: 1,200,000 | Events: 3,636,421

DONE
Abandoned journeys: 1,200,000
Product events created: 3,636,421
File: ..\data\abandoned_product_events.csv


In [20]:
from pathlib import Path

for file in DATA_DIR.glob("*.csv"):
    print(file.name)

events.csv
gaming_sessions.csv
payments.csv
product_events.csv
product_events_with_journey.csv
subscriptions.csv
users.csv


In [21]:
# Load valid game IDs from existing gaming sessions

game_ids = (
    pd.read_csv(
        DATA_DIR / "gaming_sessions.csv",
        usecols=["game_id"]
    )["game_id"]
    .dropna()
    .unique()
)

print("Unique games:", len(game_ids))
print("Sample game IDs:", game_ids[:10])

Unique games: 500
Sample game IDs: [132 171 230   5   4 115 181 127 162  19]


In [23]:
# ============================================================
# QA — ABANDONED PRODUCT JOURNEYS
# ============================================================

FILE = DATA_DIR / "abandoned_product_events.csv"

sample = pd.read_csv(FILE, nrows=500_000)

print("Sample rows:", f"{len(sample):,}")

print("\nMissing values:")
print(sample.isna().sum())

print("\nDuplicate event_id:")
print(sample["event_id"].duplicated().sum())

print("\nJourney ID range:")
print(
    int(sample["journey_id"].min()),
    "->",
    int(sample["journey_id"].max())
)

print("\nEvent ID range:")
print(
    int(sample["event_id"].min()),
    "->",
    int(sample["event_id"].max())
)

# Abandoned journeys MUST NOT have gaming sessions
print("\nRows with gaming_session_id:")
print(sample["gaming_session_id"].notna().sum())

# One journey = one user
journey_users = (
    sample.groupby("journey_id")["user_id"]
    .nunique()
)

print("\nJourneys linked to >1 user:")
print((journey_users > 1).sum())

# One journey = one device
journey_devices = (
    sample.groupby("journey_id")["device"]
    .nunique()
)

print("\nJourneys linked to >1 device:")
print((journey_devices > 1).sum())

print("\nEvent distribution:")
print(sample["event_type"].value_counts())

# Funnel counts
funnel = (
    sample.groupby("event_type")["journey_id"]
    .nunique()
    .reindex([
        "app_open",
        "login",
        "game_search",
        "game_view"
    ])
)

print("\nFunnel:")
print(funnel)

print("\nSample:")
display(sample.head(15))

Sample rows: 500,000

Missing values:
event_id                  0
user_id                   0
event_timestamp           0
event_type                0
journey_id                0
gaming_session_id    500000
game_id              310279
device                    0
dtype: int64

Duplicate event_id:
0

Journey ID range:
4000001 -> 4165000

Event ID range:
25119779 -> 25619778

Rows with gaming_session_id:
0

Journeys linked to >1 user:
0

Journeys linked to >1 device:
0

Event distribution:
event_type
app_open       165000
login          145279
game_search    115498
game_view       74223
Name: count, dtype: int64

Funnel:
event_type
app_open       165000
login          145279
game_search    115498
game_view       74223
Name: journey_id, dtype: int64

Sample:


,event_id,user_id,event_timestamp,event_type,journey_id,gaming_session_id,game_id,device
0,25119779,13388,2026-02-17 00:02:21,app_open,4000001,NaN,NaN,browser
1,25119780,13388,2026-02-17 00:03:09,login,4000001,NaN,NaN,browser
2,25119781,13388,2026-02-17 00:04:42,game_search,4000001,NaN,210.0,browser
3,25119782,116094,2025-04-11 20:20:18,app_open,4000002,NaN,NaN,macos
4,25119783,116094,2025-04-11 20:21:38,login,4000002,NaN,NaN,macos
5,25119784,116094,2025-04-11 20:24:06,game_search,4000002,NaN,112.0,macos
6,25119785,98186,2024-05-14 07:13:24,app_open,4000003,NaN,NaN,android
7,25119786,98186,2024-05-14 07:14:10,login,4000003,NaN,NaN,android
8,25119787,98186,2024-05-14 07:14:42,game_search,4000003,NaN,233.0,android
9,25119788,98186,2024-05-14 07:19:26,game_view,4000003,NaN,233.0,android


In [24]:
from pathlib import Path

for file in DATA_DIR.glob("*.csv"):
    print(file.name)

abandoned_product_events.csv
events.csv
gaming_sessions.csv
payments.csv
product_events.csv
product_events_with_journey.csv
subscriptions.csv
users.csv
